In [41]:
filepath="Students_data.csv"

In [42]:
import pandas as pd
df=pd.read_csv(filepath)

In [43]:
df.columns

Index(['Student_ID', 'Study_Hours', 'Age', 'Attendance', 'Family_Income',
       'Internet_Usage_Hours', 'Exam_Score'],
      dtype='str')

In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Student_ID            100 non-null    int64  
 1   Study_Hours           100 non-null    float64
 2   Age                   100 non-null    int64  
 3   Attendance            97 non-null     float64
 4   Family_Income         100 non-null    float64
 5   Internet_Usage_Hours  98 non-null     float64
 6   Exam_Score            100 non-null    float64
dtypes: float64(5), int64(2)
memory usage: 5.6 KB


In [45]:
df.isna().sum()

Student_ID              0
Study_Hours             0
Age                     0
Attendance              3
Family_Income           0
Internet_Usage_Hours    2
Exam_Score              0
dtype: int64

In [46]:
cols=df.columns.tolist()
cols


['Student_ID',
 'Study_Hours',
 'Age',
 'Attendance',
 'Family_Income',
 'Internet_Usage_Hours',
 'Exam_Score']

In [47]:
numerical_values=['Student_ID',
 'Study_Hours',
 'Age',
 'Attendance',
 'Family_Income',
 'Internet_Usage_Hours',
 'Exam_Score']

In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  StandardScaler
trf1=ColumnTransformer([(
    'impute', SimpleImputer(strategy='most_frequent'),['Attendance','Internet_Usage_Hours'])],
     remainder='passthrough',
         verbose_feature_names_out=False
     ).set_output(transform='pandas')

trf2=ColumnTransformer([(
    'scaler', StandardScaler(),(numerical_values))],
     remainder='passthrough',   
    verbose_feature_names_out=False
     ).set_output(transform='pandas')

    
   


In [49]:
from sklearn.pipeline import Pipeline
pipe=Pipeline([ 
    
    ('trf1',trf1),
    ('trf2',trf2)
])

In [50]:
df_t1=pipe.fit_transform(df)
df_t1.head()

,Student_ID,Study_Hours,Age,Attendance,Family_Income,Internet_Usage_Hours,Exam_Score
0,-1.714816,0.389868,1.397102,-1.381752,-0.371234,0.782746,0.050094
1,-1.680173,-0.347587,1.397102,-0.951939,-0.278586,2.558813,-0.813657
2,-1.645531,-0.441817,0.977552,0.467422,1.179831,1.314502,-1.493002
3,-1.610888,-0.441817,-0.700649,0.466445,-0.652141,-0.684903,0.396290
4,-1.576245,2.237602,-1.539749,0.467422,0.095441,-0.291403,-0.605765


In [51]:
from sklearn.base import BaseEstimator, TransformerMixin
class FeatureEngineering(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        X["study_attendance"] = (
            X["Study_Hours"] * X["Attendance"]
        )

        X["internet_study_ratio"] = (
            X["Internet_Usage_Hours"] /
            (X["Study_Hours"] + 1)
        )

        X["study_attendance_ratio"] = (
            X["Study_Hours"] /
            (X["Attendance"] + 1)
        )

        X["Age_Group"] = pd.cut(
            X["Age"],
            bins=[0, 18, 22, 26, 100],
            labels=["Young", "Adult", "Older", "Very_Older"]
        )

        X["Attendance_Level"] = pd.cut(
            X["Attendance"],
            bins=[0, 70, 85, 100],
            labels=["Low", "Medium", "High"]
        )

        return X

In [52]:
from sklearn.pipeline import Pipeline
pipe=Pipeline([
    ("feature_engineering", FeatureEngineering())
])

In [53]:
feature_engineered_df = pipe.fit_transform(df)
feature_engineered_df

,Student_ID,Study_Hours,Age,Attendance,Family_Income,Internet_Usage_Hours,Exam_Score,study_attendance,internet_study_ratio,study_attendance_ratio,Age_Group,Attendance_Level
0,1,4.79,25,68.22,26401.83,5.13,68.46,326.7738,0.886010,0.069200,Older,Low
1,2,2.99,25,72.62,29899.43,10.14,58.53,217.1338,2.541353,0.040614,Older,Medium
2,3,2.76,24,87.15,84956.82,6.63,50.72,240.5340,1.763298,0.031310,Older,High
3,4,2.76,20,87.14,15797.16,0.99,72.44,240.5064,0.263298,0.031314,Adult,High
4,5,9.30,18,87.15,44019.48,2.10,60.92,810.4950,0.203883,0.105502,Young,High
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,1.81,25,88.30,81963.94,1.02,43.47,159.8230,0.362989,0.020269,Older,High
96,97,10.84,18,73.71,6406.16,4.37,72.83,799.0164,0.369088,0.145094,Young,Medium
97,98,4.71,18,76.40,15686.61,7.90,79.25,359.8440,1.383538,0.060853,Young,Medium
98,99,1.40,20,89.47,27030.45,2.51,52.26,125.2580,1.045833,0.015475,Adult,High
